# ERS 1386 MIRI data download for the spaceKLIP tutorial

This notebook downloads the public MIRI `uncal.fits` products used by the spaceKLIP MIRI reduction tutorial without requiring a MAST API token.

The original tutorial uses `jwst_download.py`. Here, the query is done with `spaceKLIP.mast.query_coron_datasets`, and the public files are downloaded with `astroquery.mast.Observations.download_file`.

Reference tutorial:

https://spaceklip.readthedocs.io/en/latest/tutorials/tutorial_MIRI_reductions.html

## Setup and imports

In [1]:
import os
from pathlib import Path

import numpy as np
import subprocess

import spaceKLIP

import matplotlib
matplotlib.rcParams.update({"font.size": 14})
%matplotlib inline

from astropy.io import fits
from astropy.table import unique
from astroquery.mast import Observations

import webbpsf_ext
webbpsf_ext.setup_logging("WARN", verbose=False)

## Original tutorial download command

The MIRI tutorial uses the following `jwst_download.py` command. This may require a MAST API token in some environments, even for public data, depending on the installed `jwst_mast_query` behavior.

This notebook replaces the command below with a token-free public-data workflow.

In [3]:
# This is the original tutorial-style command.
# Keep this cell for reference only; it is not needed for the token-free workflow below.

data_root = "data_miri_hd65426"

# Make subdirectories to put the data in.

os.makedirs(data_root, exist_ok=True)
os.makedirs(os.path.join(data_root, 'uncal'), exist_ok=True)

download_cmd = (
    "yes | jwst_download.py --propID 1386 -i miri -l 1200 "
    "--obsnums 4 5 6 7 8 9 28 29 30 31 "
    "--outsubdir data_miri_hd65426/uncal --skip_propID2outsubdir "
    "-f uncal --date_select 59371.0+"
)

process=subprocess.Popen(download_cmd, shell=True,
                         stdout=subprocess.PIPE,
                         stderr=subprocess.PIPE)
stdout, stderr = process.communicate()

# Uncomment to print the download log and any errors.
print(stdout.decode())
print(stderr.decode())

INSTRUMENT:  miri
obsmode:  [None]
propID:  01386
obsnums:  [4, 5, 6, 7, 8, 9, 28, 29, 30, 31]

Traceback (most recent call last):
  File "/usr/local/anaconda3/envs/spaceklip/bin/jwst_download.py", line 8, in <module>
    sys.exit(main())
             ^^^^^^
  File "/usr/local/anaconda3/envs/spaceklip/lib/python3.11/site-packages/jwst_mast_query/scripts/jwst_download.py", line 23, in main
    download.login(raiseErrorFlag=True)
  File "/usr/local/anaconda3/envs/spaceklip/lib/python3.11/site-packages/jwst_mast_query/jwst_query.py", line 471, in login
    raise RuntimeError("No token!! Cannot login! you can set the token with \$API_MAST_TOKEN, as 'token' in the config file, or with --token ")
RuntimeError: No token!! Cannot login! you can set the token with \$API_MAST_TOKEN, as 'token' in the config file, or with --token 



### using SpaceKLIP.mast.download_files
### now spaceKLIP requires the API TOKEN even if you want to download public data

reference: https://spaceklip.readthedocs.io/en/latest/tutorials/MAST%20query%20tools%20for%20coronagraphic%20datasets.html

In [5]:
table = spaceKLIP.mast.query_coron_datasets('MIRI', program=1386, filt='F1550C', kind='SCI', return_filenames=True,
                                    level='uncal')

spaceKLIP.mast.download_files(table)

ValueError: Must define MAST_API_TOKEN env variable or specify mast_api_token parameter

## Token-free public-data workflow

Use `spaceKLIP.mast.query_coron_datasets` to find the same MIRI ERS 1386 products, then download the public files directly using their MAST product URIs.

The query below intentionally mirrors the original tutorial command:

```text
--propID 1386
-i miri
--obsnums 4 5 6 7 8 9 28 29 30 31
-f uncal
--date_select 59371.0+
--outsubdir data_miri_hd65426/uncal
```

In [6]:
# Output directory for this tutorial.
data_root = Path("data_miri_hd65426")
uncal_dir = data_root / "uncal"
uncal_dir.mkdir(parents=True, exist_ok=True)

# Query settings matching the MIRI tutorial.
program = 1386
obsnums = [4, 5, 6, 7, 8, 9, 28, 29, 30, 31]

# Set this to "F1550C" if you only want the filter used later in the tutorial.
# Leave as None to reproduce the original download command more closely.
download_filter = None

table = spaceKLIP.mast.query_coron_datasets(
    "MIRI",                       # inst:
                                  #   Instrument name. Here we use MIRI.

    filt=download_filter,          # filt:
                                  #   Optional MIRI filter selection, e.g. "F1550C".
                                  #   None means no filter selection, matching the
                                  #   original jwst_download.py command.

    program=program,               # program:
                                  #   JWST Program ID.
                                  #   This corresponds to --propID 1386.

    obsnum=obsnums,                # obsnum:
                                  #   Observation numbers within the JWST program.
                                  #   This corresponds to:
                                  #   --obsnums 4 5 6 7 8 9 28 29 30 31.

    return_filenames=True,         # return_filenames:
                                  #   Return individual product filenames, not just
                                  #   one summary row per observation.

    level="uncal",                 # level:
                                  #   Return *_uncal.fits product filenames.
                                  #   This corresponds to -f uncal.

    ignore_exclusive_access=True,  # ignore_exclusive_access:
                                  #   Exclude restricted / exclusive-access data.
                                  #   This keeps the workflow token-free for public data.
)

# Mimic --date_select 59371.0+
# The original jwst_download.py date selection is based on observation date.
# The spaceKLIP query table exposes vststart_mjd, so we apply the cut here.
table = table[table["vststart_mjd"] >= 59371.0]

# Remove any duplicate filenames just in case the query returns duplicated rows.
table = unique(table, keys="filename")

print(f"Found {len(table)} public MIRI uncal files.")
table["filename", "kind", "filter", "coronmsk", "targname", "obslabel", "visit_id", "vststart_mjd", "isRestricted"]

Found 30 public MIRI uncal files.


filename,kind,filter,coronmsk,targname,obslabel,visit_id,vststart_mjd,isRestricted
str45,bytes3,str6,str9,str9,str22,str12,float64,bool
jw01386004001_04101_00001_mirimage_uncal.fits,SCI,F1140C,4QPM_1140,HD 116434,MIRI 1140C - Roll 2,V01386004001,59777.74378645833,False
jw01386005001_04101_00001_mirimage_uncal.fits,SCI,F1140C,4QPM_1140,HD 116434,MIRI 1140C - Roll 1,V01386005001,59777.78972418982,False
jw01386006001_04101_00001_mirimage_uncal.fits,REF,F1140C,4QPM_1140,* phi Cen,MIRI 1140C - REF,V01386006001,59777.82637009259,False
jw01386006001_04101_00002_mirimage_uncal.fits,REF,F1140C,4QPM_1140,* phi Cen,MIRI 1140C - REF,V01386006001,59777.82637009259,False
jw01386006001_04101_00003_mirimage_uncal.fits,REF,F1140C,4QPM_1140,* phi Cen,MIRI 1140C - REF,V01386006001,59777.82637009259,False
jw01386006001_04101_00004_mirimage_uncal.fits,REF,F1140C,4QPM_1140,* phi Cen,MIRI 1140C - REF,V01386006001,59777.82637009259,False
jw01386006001_04101_00005_mirimage_uncal.fits,REF,F1140C,4QPM_1140,* phi Cen,MIRI 1140C - REF,V01386006001,59777.82637009259,False
jw01386006001_04101_00006_mirimage_uncal.fits,REF,F1140C,4QPM_1140,* phi Cen,MIRI 1140C - REF,V01386006001,59777.82637009259,False
jw01386006001_04101_00007_mirimage_uncal.fits,REF,F1140C,4QPM_1140,* phi Cen,MIRI 1140C - REF,V01386006001,59777.82637009259,False


## Download the public files via astroquery

This cell downloads each file into:

```text
data_miri_hd65426/uncal/
```

If a file already exists locally, it is skipped.

In [7]:
for row in table:
    filename = str(row["filename"])
    uri = f"mast:JWST/product/{filename}"
    local_path = uncal_dir / filename

    if local_path.exists():
        print(f"Already exists: {local_path}")
        continue

    print(f"Downloading {filename}")
    result = Observations.download_file(
        uri,
        local_path=str(local_path),
        cache=False,
    )
    print(result)

('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)
('COMPLETE', None, None)


## Verify the downloaded files

In [8]:
downloaded = sorted(uncal_dir.glob("*_uncal.fits"))

print(f"Downloaded / available uncal files: {len(downloaded)}")
for path in downloaded[:20]:
    print(path.name)

if len(downloaded) > 20:
    print(f"... and {len(downloaded) - 20} more")

Downloaded / available uncal files: 30
jw01386004001_04101_00001_mirimage_uncal.fits
jw01386005001_04101_00001_mirimage_uncal.fits
jw01386006001_04101_00001_mirimage_uncal.fits
jw01386006001_04101_00002_mirimage_uncal.fits
jw01386006001_04101_00003_mirimage_uncal.fits
jw01386006001_04101_00004_mirimage_uncal.fits
jw01386006001_04101_00005_mirimage_uncal.fits
jw01386006001_04101_00006_mirimage_uncal.fits
jw01386006001_04101_00007_mirimage_uncal.fits
jw01386006001_04101_00008_mirimage_uncal.fits
jw01386006001_04101_00009_mirimage_uncal.fits
jw01386007001_04101_00001_mirimage_uncal.fits
jw01386007001_04101_00002_mirimage_uncal.fits
jw01386007001_04101_00003_mirimage_uncal.fits
jw01386007001_04101_00004_mirimage_uncal.fits
jw01386007001_04101_00005_mirimage_uncal.fits
jw01386007001_04101_00006_mirimage_uncal.fits
jw01386007001_04101_00007_mirimage_uncal.fits
jw01386007001_04101_00008_mirimage_uncal.fits
jw01386007001_04101_00009_mirimage_uncal.fits
... and 10 more


## Return to the MIRI spaceKLIP tutorial

The MIRI reduction tutorial next creates a `spaceKLIP` database from the downloaded `uncal` files.

The tutorial reduces one filter by default:

```python
filt = "F1550C"
```

Set `filt = None` if you intentionally want to index all downloaded filters.

In [9]:
program = 1386
filt = "F1550C"  # Set to None to disable filter selection and load all filters.

database = spaceKLIP.database.create_database(
    input_dir=os.path.join(data_root, "uncal"),
    output_dir=data_root,
    assoc_using_targname=False,
    filt=filt,
    pid=program,
)

[spaceKLIP.database:INFO] --> Identified 1 concatenation(s)
[spaceKLIP.database:INFO]   --> Concatenation 1: JWST_MIRI_MIRIMAGE_F1550C_NONE_4QPM_1550_MASK1550
 TYPE  EXP_TYPE DATAMODL TELESCOP ...      ROLL_REF      BLURFWHM NANMASKFILE
------ -------- -------- -------- ... ------------------ -------- -----------
   SCI MIR_4QPM   STAGE0     JWST ... 108.17107381748745      nan        NONE
   SCI MIR_4QPM   STAGE0     JWST ... 117.54983150698699      nan        NONE
SCI_BG MIR_4QPM   STAGE0     JWST ... 112.74286239818893      nan        NONE
SCI_BG MIR_4QPM   STAGE0     JWST ... 112.74285728609199      nan        NONE
REF_BG MIR_4QPM   STAGE0     JWST ... 112.79986264216615      nan        NONE
REF_BG MIR_4QPM   STAGE0     JWST ... 112.79993618079723      nan        NONE
   REF MIR_4QPM   STAGE0     JWST ...  109.2146450046046      nan        NONE
   REF MIR_4QPM   STAGE0     JWST ... 109.21463437851206      nan        NONE
   REF MIR_4QPM   STAGE0     JWST ... 109.21463905922128     

In [10]:
database.summarize()

MIRI_F1550C_4QPM
	STAGE0: 15 files;	2 SCI, 9 REF, 4 BG


## Optional: index all downloaded MIRI filters

Use this only if you downloaded all filters and want to inspect the full local dataset.

In [11]:
program = 1386
filt = None

database_all = spaceKLIP.database.create_database(
    input_dir=os.path.join(data_root, "uncal"),
    output_dir=data_root,
    assoc_using_targname=False,
    filt=filt,
    pid=program,
)

database_all.summarize()

[spaceKLIP.database:INFO] --> Identified 2 concatenation(s)
[spaceKLIP.database:INFO]   --> Concatenation 1: JWST_MIRI_MIRIMAGE_F1140C_NONE_4QPM_1140_MASK1140
 TYPE  EXP_TYPE DATAMODL TELESCOP ...      ROLL_REF      BLURFWHM NANMASKFILE
------ -------- -------- -------- ... ------------------ -------- -----------
   SCI MIR_4QPM   STAGE0     JWST ... 108.04884661017272      nan        NONE
   SCI MIR_4QPM   STAGE0     JWST ... 117.42688016738536      nan        NONE
SCI_BG MIR_4QPM   STAGE0     JWST ... 112.71126058935718      nan        NONE
SCI_BG MIR_4QPM   STAGE0     JWST ... 112.71125619540183      nan        NONE
REF_BG MIR_4QPM   STAGE0     JWST ... 112.73537192949475      nan        NONE
REF_BG MIR_4QPM   STAGE0     JWST ... 112.73532731628123      nan        NONE
   REF MIR_4QPM   STAGE0     JWST ... 109.19560580273766      nan        NONE
   REF MIR_4QPM   STAGE0     JWST ... 109.19561133977358      nan        NONE
   REF MIR_4QPM   STAGE0     JWST ... 109.19559340271205     